# Odin's Eye — Colab Runner (A100)

Runs Phases 0–2 of the multi-camera tracking pipeline.  
Requires: **A100 GPU runtime**, Google Drive for YOLO weights + output persistence.

**Before running:** Fill in your Kaggle credentials in cell 2 and YOLO weights Drive path in cell 3.

## 0. Clone repo & install core package

In [10]:
%cd /content
!rm -rf Odin-s-Eye
!git clone https://github.com/ozi14/Odin-s-Eye.git
%cd Odin-s-Eye
%pip install -q -e .

/content
Cloning into 'Odin-s-Eye'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (56/56), done. (17/56)
remote: Total 78 (delta 17), reused 74 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 88.47 KiB | 3.28 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Odin-s-Eye
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for odin-eye (pyproject.toml) ... done


## 1. Install D4SM + SAM2.1 (the critical step)

In [11]:
%%bash
# ── Clone d4sm + download checkpoint (no pip here) ──
cd /content/Odin-s-Eye

if [ ! -d "external/d4sm" ]; then
    echo "[1/3] Cloning D4SM..."
    mkdir -p external
    git clone --depth 1 https://github.com/alanlukezic/d4sm.git external/d4sm
else
    echo "[1/3] D4SM already cloned."
fi

if [ ! -f "external/d4sm/setup.py" ]; then
    echo "FATAL: external/d4sm/setup.py not found"
    exit 1
fi
echo "[1/3] OK: setup.py found"

mkdir -p checkpoints
if [ ! -f "checkpoints/sam2.1_hiera_large.pt" ]; then
    echo "[2/3] Downloading SAM2.1-Large checkpoint (~900MB)..."
    wget -q --show-progress -P checkpoints/ \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
else
    echo "[2/3] Checkpoint already exists."
fi

echo "[3/3] Clone + checkpoint ready."

[1/3] Cloning D4SM...
[1/3] OK: setup.py found
[2/3] Downloading SAM2.1-Large checkpoint (~900MB)...
[3/3] Clone + checkpoint ready.


Cloning into 'external/d4sm'...

     0K .......... .......... .......... .......... ..........  0% 3.68M 3m53s
    50K .......... .......... .......... .......... ..........  0% 4.91M 3m23s
   100K .......... .......... .......... .......... ..........  0% 13.2M 2m37s
   150K .......... .......... .......... .......... ..........  0% 12.3M 2m15s
   200K .......... .......... .......... .......... ..........  0% 14.8M 2m0s
   250K .......... .......... .......... .......... ..........  0% 26.4M 1m45s
   300K .......... .......... .......... .......... ..........  0% 16.0M 98s
   350K .......... .......... .......... .......... ..........  0% 17.7M 92s
   400K .......... .......... .......... .......... ..........  0% 21.5M 86s
   450K .......... .......... .......... .......... ..........  0% 29.3M 80s
   500K .......... .......... .......... .......... ..........  0% 58.8M 74s
   550K .......... .......... .......... .......... ..........  0% 31.4M 70s
   600K .......... .......... ..

In [12]:
# ── Install sam2 via %pip so the Jupyter kernel picks it up ──
# (This is why setup was split: !pip inside %%bash doesn't update the running kernel)
%pip install -e /content/Odin-s-Eye/external/d4sm/
%pip install -q hydra-core omegaconf ultralytics

Obtaining file:///content/Odin-s-Eye/external/d4sm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp312-cp312-linux_x86_64.whl size=9299 sha256=061ebad0881f108cd00e7d834d7370e5513d61c67e798741231245b81f1deda3
  Stored in directory: /tmp/pip-ephem-wheel-cache-s50ungaw/wheels/3f/29/4c/c615457349a9507cc84479a84fa1cd3b14cf06598674ec0cd3
Successfully built SAM-2
  Attempting uninstall: SAM-2
    Found existing installation: SAM-2 1.0
    Uninstalling SAM-2-1.0:
      Successfully uninstalled SAM-2-1.0


In [13]:
# ── Build CUDA extensions (optional, improves postprocessing speed) ──
%cd /content/Odin-s-Eye/external/d4sm
!python setup.py build_ext --inplace
%cd /content/Odin-s-Eye

/content/Odin-s-Eye/external/d4sm
running build_ext
W0404 02:20:43.469000 39451 torch/utils/cpp_extension.py:659] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W0404 02:20:43.486000 39451 torch/utils/cpp_extension.py:535] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
building 'sam2._C' extension
creating build/temp.linux-x86_64-cpython-312/sam2/csrc
/usr/local/cuda/bin/nvcc -I/usr/local/lib/python3.12/dist-packages/torch/include -I/usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.12 -c sam2/csrc/connected_components.cu -o build/temp.linux-x86_64-cpython-312/sam2/csrc/connected_components.o -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr --compiler-options '-fPIC' -DCUDA_HAS_FP16=1 -D__CUDA_NO_

In [ ]:
import os
os._exit(0)  # hard-kill the kernel; Colab auto-restarts it

: 

: 

: 

In [1]:
# ── Gate check: if this cell fails, STOP and fix setup_tracking_v2.sh ──
import sam2
print("sam2 OK:", sam2.__file__)

from sam2.utils.misc import fill_holes_in_mask_scores
print("fill_holes_in_mask_scores OK")

import torch
print(f"CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}")

sam2 OK: /content/Odin-s-Eye/external/d4sm/sam2/__init__.py
fill_holes_in_mask_scores OK
CUDA: True, device: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Download WILDTRACK dataset

In [2]:
import os

# ╔══════════════════════════════════════════════════════╗
# ║  FILL THESE IN — then clear key before committing   ║
# ╚══════════════════════════════════════════════════════╝
KAGGLE_USERNAME = "ouzhanuar"
KAGGLE_KEY      = "KGAT_1f5f4b2da14780fb5735c25e7443e653"                                       # ← paste key here

assert KAGGLE_KEY, "Paste your Kaggle API key above"

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write('{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
os.chmod("/root/.kaggle/kaggle.json", 0o600)

%cd /content/Odin-s-Eye
!python scripts/data_prep/05_download_wildtrack.py

/content/Odin-s-Eye
Using Colab cache for faster access to the 'large-scale-multicamera-detection-dataset' dataset.

Path to dataset files: /kaggle/input/large-scale-multicamera-detection-dataset


In [3]:
!ls /kaggle/input/large-scale-multicamera-detection-dataset/Wildtrack/

annotations_positions  cam2.mp4  cam5.mp4  Image_subsets
calibrations	       cam3.mp4  cam6.mp4  rectangles.pom
cam1.mp4	       cam4.mp4  cam7.mp4


In [4]:
import os
from pathlib import Path

REPO = Path("/content/Odin-s-Eye")
SRC = Path("/kaggle/input/large-scale-multicamera-detection-dataset/Wildtrack/")
DST = REPO / "datasets" / "Wildtrack"

assert (SRC / "Image_subsets").is_dir(), f"Missing Image_subsets in {SRC}"

DST.parent.mkdir(parents=True, exist_ok=True)
if DST.exists() or DST.is_symlink():
    if DST.is_symlink():
        DST.unlink()
    elif DST.is_dir():
        import shutil; shutil.rmtree(DST)
os.symlink(SRC.resolve(), DST, target_is_directory=True)

print(f"C1 frames: {len(list((DST / 'Image_subsets' / 'C1').glob('*.png')))}")
print(f"Calib dirs: {sorted(x.name for x in (DST / 'calibrations').iterdir())}")

C1 frames: 401
Calib dirs: ['extrinsic', 'intrinsic_original', 'intrinsic_zero']


## 3. Mount Drive & copy YOLO weights

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import shutil
from pathlib import Path

# ╔══════════════════════════════════════════════════════╗
# ║  UPDATE THIS to your actual best.pt path on Drive   ║
# ╚══════════════════════════════════════════════════════╝
DRIVE_PT = Path("/content/drive/MyDrive/Cv_project_v1/models/yolo26m_ft_v12/weights/best.pt")

assert DRIVE_PT.exists(), f"YOLO weights not found: {DRIVE_PT}"

dst = Path("/content/Odin-s-Eye/models/yolo26m_ft_v1/weights/best.pt")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_PT, dst)
print(f"Copied {dst.stat().st_size / 1e6:.1f} MB -> {dst}")

Copied 44.1 MB -> /content/Odin-s-Eye/models/yolo26m_ft_v1/weights/best.pt


## 4. Phase 0 — Calibration

In [7]:
%cd /content/Odin-s-Eye
!python scripts/pipeline/09_calibration.py

/content/Odin-s-Eye
  PHASE 0: Offline Calibration Setup

📁 Step 1: Loading calibration files...
✅ Loaded calibrations for 7 cameras

📐 Step 2: Computing projection matrices and homographies...
  C1: det(H)=-873967091.14, fx=1743.4, fy=1735.2, tvec=[-526, 45, 987]
  C2: det(H)=-585189186.02, fx=1707.3, fy=1719.0, tvec=[1195, -337, 2041]
  C3: det(H)=-807071894.87, fx=1738.7, fy=1752.9, tvec=[55, -213, 1993]
  C4: det(H)=-822535343.94, fx=1725.3, fy=1720.6, tvec=[42, -45, 1107]
  C5: det(H)=-499407238.39, fx=1708.7, fy=1737.2, tvec=[837, 86, 600]
  C6: det(H)=-683362500.60, fx=1743.0, fy=1746.0, tvec=[-339, 63, 1044]
  C7: det(H)=-1033873107.37, fx=1732.5, fy=1757.6, tvec=[-649, -57, 1053]

🗺️ Step 3: Computing FOV polygons on ground plane...
  C1: area=143.4 m², bounds=(-300, -90) → (950, 1110) cm
  C2: area=367.5 m², bounds=(-25, -90) → (3300, 1110) cm
  C3: area=150.2 m², bounds=(-300, -90) → (975, 1110) cm
  C4: area=66.9 m², bounds=(-300, -90) → (625, 1110) cm
  C5: area=375.0 m², 

## 5. Phase 1 — Local tracking (SAM2.1 + D4SM + DINOv2)

Run 5-frame smoke test first. If it passes, uncomment the full run.

In [9]:
%cd /content/Odin-s-Eye
!git pull origin main

/content/Odin-s-Eye
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 965 bytes | 965.00 KiB/s, done.
From https://github.com/ozi14/Odin-s-Eye
 * branch            main       -> FETCH_HEAD
   f8dcdc1..68970f9  main       -> origin/main
Updating f8dcdc1..68970f9
Fast-forward
 odin_eye/tracking/dam4sam_tracker.py    |  2 ++
 scripts/pipeline/10_local_tracker_v2.py | 19 ++++++++++++++++---
 2 files changed, 18 insertions(+), 3 deletions(-)


In [10]:
%cd /content/Odin-s-Eye
!python scripts/pipeline/10_local_tracker_v2.py \
    --device cuda \
    --max_frames 5

/content/Odin-s-Eye
Loading Phase 0 calibration cache...
Found 401 synchronised frames
Loading YOLO detector (/content/Odin-s-Eye/models/yolo26m_ft_v1/weights/best.pt)...
Loading SAM2.1-Large on cuda...
D4SM engine ready. Memory storage: CPU
Loading DINOv2 ReID extractor (dinov2_vitl14_reg) on cuda...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
DINOv2 ReID ready — 1024-D features
MultiCameraTrackerV2 ready — 7 c